# Data Trim

This notebook trims the data by removing nans etc

In [ ]:
import sys
sys.path.append('../src')

In [ ]:
import os
import pandas as pd
from pprint import pprint
from util_IO import (
    load_pickle_from_main_project_dir,
    EDA_dirs_structure,
    load_attributes_df,
    load_timeseries_df
)

# Set pandas to display a maximum of 300 columns
pd.set_option('display.max_columns', 300)
pd.set_option('display.max_rows', 1000)

## Parameters

### Load metadata from previous step

In [ ]:
aggr_parameters_dict, camels_gb_use_case_dir = load_pickle_from_main_project_dir(
    'aggr_parameters_dict.pkl'
)

### Retrieve variables in use

In [ ]:
camels_gb_data_attributes_aggr_dir = aggr_parameters_dict['camels_gb_data_attributes_aggr_dir']
camels_gb_data_timeseries_aggr_dir = aggr_parameters_dict['camels_gb_data_timeseries_aggr_dir']
attributes_index = aggr_parameters_dict["attributes"]["attributes_index"]
date_field = aggr_parameters_dict["timeseries"]["date_field"]
attributes_fundamental_fields = aggr_parameters_dict['attributes']['aggregations']['fundamental']

# Retrieve aggregated file

In [ ]:
# Attributes
attributes_df = load_attributes_df(
    camels_gb_data_attributes_aggr_dir,
    "fundamental.csv",
    attributes_index
)

display(attributes_df.head(3))

# Timeseries
timeseries_df = load_timeseries_df(
    camels_gb_data_timeseries_aggr_dir,
    "timeseries.csv",
    date_field
)

display(timeseries_df.head(3))

# Columns removal

## Columns removal for ***attributes***

As shown in ***02a-EDA-Attributes***, the only dataset with `NaN` values is the one related to ***attributes***. Columns to remove are:
 - **bankfull_flow**

In [ ]:
# Define columns to be removed
columns_to_remove = [
    'bankfull_flow'
]

In [ ]:
# Reduce the columns to the dimensions to use in the model
attributes_df = (
    attributes_df[
        [col for sublist in attributes_fundamental_fields.values() for col in sublist]
    ]
)

# Removing columns because of NaN
attributes_postEDA_df = (
    attributes_df
        .drop(
            columns=columns_to_remove
        )
)

## Column removal for ***timeseries***

In [ ]:
# Define columns to be removed
columns_to_remove = [
    'pet',
    'peti',
    'discharge_spec',
    'discharge_vol_files',
    'date_diff',
    'date_consecutive_day'
]

# Removing columns
timeseries_postEDA_df = (
    timeseries_df
        .drop(
            columns=columns_to_remove
        )
)

# Rows removal

## ***attributes*** with `NaN`

In [ ]:
# Identify rows with at least one NaN value
catchmentsID_with_nan_list = attributes_postEDA_df[attributes_postEDA_df.isna().any(axis=1)].index.to_list()

print(catchmentsID_with_nan_list)

In [ ]:
# Drop rows with at least one NaN value
attributes_postEDA_df.dropna(
    inplace=True
)

## Catchments removal because not in common

In [ ]:
# Sets definitions
attributes_set = set(attributes_postEDA_df.index)
timeseries_set = set(timeseries_postEDA_df['catchmentID'])

### Catchments in ***attributes***, but NOT in ***timeseries***

In [ ]:
# Identify elements present in attributes but not in timeseries
attributes_not_in_timeseries = attributes_set - timeseries_set

print(attributes_not_in_timeseries)

### Catchments in timeseries, but NOT in attributes

In [ ]:
# Identify elements present in timeseries but not in attributes
timeseries_not_in_attributes = timeseries_set - attributes_set

print(timeseries_not_in_attributes)

### Filter out for common catchments only

In [ ]:
# Identify common elements
common_elements = list(attributes_set & timeseries_set)

# Filter attributes to include only common elements
attributes_postEDA_df = (
    attributes_postEDA_df
        .loc[common_elements]
)

# Filter timeseries_df to include only common elements
timeseries_postEDA_df = (
    timeseries_postEDA_df[
        timeseries_postEDA_df['catchmentID']
            .isin(common_elements)
    ]
)

## Check on `catchmentID` coherence

In [ ]:
# Check
assert set(timeseries_postEDA_df['catchmentID'].unique()) == set(attributes_postEDA_df.index), (
    "Catchment IDs for the two datasets are NOT corresponding"
)

# Save

In [ ]:
# _____________________
# attributes_postEDA_df

# Define path to save
path = os.path.join(
        camels_gb_data_attributes_aggr_dir,
        "fundamental_postEDA.csv"
)

# Save
attributes_postEDA_df.to_csv(path)

# _____________________
# timeseries_postEDA_df

# Define path to save
path = os.path.join(
        camels_gb_data_timeseries_aggr_dir,
        "timeseries_postEDA.csv"
)

# Save
timeseries_postEDA_df.to_csv(
    path,
    index=False
)